# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet')

In [4]:
X_train.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
0,0.50,0.203815,0.328482,0.154734,1.585901,-0.308849,1.487008,-1.486487,1.666667,1.040769,...,0.947453,0.005655,0.020178,0.000549,0.997580,0.001872,0.012740,0.40,0,0.781831
1,0.50,0.199575,0.327181,0.176114,0.229628,-1.066602,0.033534,1.440545,-1.000000,-5.009333,...,0.074107,0.081854,0.098684,0.015175,0.007697,0.977128,0.013316,0.20,1,0.885456
2,0.50,0.218840,0.326837,0.187526,2.116616,0.638343,1.902200,-1.486487,1.000000,-1.970285,...,0.226475,0.059038,0.614716,0.024077,0.912645,0.063278,0.014095,0.65,0,0.537300
3,0.25,0.200526,0.100888,0.145166,-1.244581,-0.498287,-1.029455,-0.510810,0.000000,0.338641,...,0.088099,0.041344,0.113803,0.950726,0.010153,0.039121,0.010598,0.35,0,0.239316
4,0.50,0.192082,0.327657,0.214557,0.170660,-1.445478,0.092589,-1.486487,1.000000,0.169816,...,0.256086,0.040890,0.083385,0.009589,0.004844,0.985566,0.009270,0.10,1,0.906308


In [5]:
X_test.head()

,ordinal_encoder__compound,target_encoder__driver,target_encoder__compound,target_encoder__race,standard_scaler__lapnumber,standard_scaler__position,standard_scaler__raceprogress,standard_scaler__year,robust_scaler__position_change,robust_scaler__cumulative_degradation,...,remainder__fuzzy_1_position_position_change,remainder__fuzzy_2_position_position_change,remainder__fuzzy_3_position_position_change,remainder__fuzzy_0_raceprogress_lapnumber,remainder__fuzzy_1_raceprogress_lapnumber,remainder__fuzzy_2_raceprogress_lapnumber,remainder__lap_time_inv,remainder__position_norm,remainder__is_pit_lap,remainder__race_progress_sin
id,,,,,,,,,,,,,,,,,,,,,
439140,0.25,0.182308,0.101132,0.133462,-0.124182,-1.066602,0.261317,-0.510810,0.000000,0.396609,...,0.001144,0.000465,0.000894,0.067702,0.028176,0.904122,0.010708,0.20,0,0.954721
439141,0.25,0.356316,0.101132,0.150547,0.052723,-1.634917,0.300590,-0.510810,0.000000,0.470778,...,0.081596,0.033933,0.053686,0.019477,0.011500,0.969023,0.011005,0.05,0,0.963550
439142,0.25,0.121796,0.101132,0.133462,0.052723,0.259466,0.489100,-0.510810,0.000000,0.301036,...,0.087683,0.072629,0.701444,0.038897,0.031448,0.929655,0.010768,0.55,0,0.992709
439143,0.00,0.198847,0.193475,0.253713,-1.008708,1.017219,-1.025511,0.464868,0.333333,0.724449,...,0.023113,0.022859,0.935769,0.984826,0.002887,0.012287,0.010530,0.75,0,0.242362
439144,0.50,0.197127,0.327536,0.114051,1.703838,0.448905,1.518343,0.464868,2.333333,0.003617,...,0.718809,0.040632,0.151701,0.002043,0.991340,0.006618,0.010090,0.60,0,0.766044


# Machine Learning

## Stacking

In [6]:
lgbm = load_pickle('../models/model_lightgbm.pkl')
cat = load_pickle('../models/model_catboost.pkl')
xgb = load_pickle('../models/model_xgboost.pkl')
lg = load_pickle('../models/model_logistic_regression.pkl')
hist = load_pickle('../models/model_histgradientboosting.pkl')

In [7]:
stacking = StackingClassifier(
    estimators=[
        ('lgbm', lgbm),
        ('cat', cat),
        ('xgb', xgb),
        ('lg', lg),
        ('hist', hist)
    ],
    final_estimator=LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=2000
    ),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False,
    verbose=0
)

In [8]:
stacking.fit(X_train, y_train.PitNextLap)

StackingClassifier(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimators=[('lgbm',
                                Pipeline(steps=[('drop_feat',
                                                 DropFeatures(features_to_drop=['remainder__stint_low',
                                                                                'remainder__lapnumber_low',
                                                                                'remainder__lapnumber_high',
                                                                                'remainder__stint_high',
                                                                                'remainder__position_high',
                                                                                'remainder__position_low',
                                                                                'remainder__is_pit_lap'])),
                                                ('lgbm',
                                                 L...
                                                                                'remainder__lapnumber_low'])),
                                                ('hist',
                                                 HistGradientBoostingClassifier(early_stopping=True,
                                                                                l2_regularization=6.70145048510531,
                                                                                learning_rate=0.07348770421656471,
                                                                                max_bins=190,
                                                                                max_depth=10,
                                                                                max_iter=1000,
                                                                                min_samples_leaf=81,
                                                                                n_iter_no_change=50,
                                                                                random_state=42))]))],
                   final_estimator=LogisticRegression(C=0.1,
                                                      class_weight='balanced',
                                                      max_iter=2000),
                   n_jobs=-1, stack_method='predict_proba')

Stacking(Voting(LightGBM, XGBoost, CatBoost, HistGradientBossting, Linear Regression)

# Submission

In [9]:
submission = pd.read_csv('../data/sample_submission.csv')

In [10]:
submission['PitNextLap'] = stacking.predict_proba(X_test)[:, 1]

In [11]:
submission.to_csv('../data/submission.csv', index=False)